In [1]:
# Preparação para o Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Altere o caminho caso o nome da pasta no seu Drive seja diferente
    caminho_pasta = '/content/drive/MyDrive/artefatos colab'
    if os.path.exists(caminho_pasta):
        os.chdir(caminho_pasta)
        print('Diretório alterado para:', os.getcwd())
    else:
        print('ATENÇÃO: Pasta não encontrada no Drive. Verifique se o nome está correto.')
except ImportError:
    print('Não está rodando no Google Colab. Mantendo diretório atual.')


Mounted at /content/drive
Diretório alterado para: /content/drive/MyDrive/artefatos colab


# Sprint 4 — Pipeline RAG e Assistente Conversacional
Neste notebook construímos o RAG sobre a documentação técnica (Sprints 1 e 2) e instanciamos o Assistente Conversacional (LLM).

In [2]:
# 1. Força a desinstalação do numpy conflitante e instala a versão 1.26.4
!pip install "numpy<2" packaging -q

# 2. Instala as bibliotecas do projeto RAG
!pip install sentence-transformers faiss-cpu "langchain<0.2.0" "langchain-community<0.2.0" "langchain-core<0.2.0" langchain-text-splitters huggingface_hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 108.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which 

In [1]:
import json
import numpy as np
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer

## 1. Chunking e Indexação (FAISS)

In [2]:
corpus_textos = [
    """# Manual do Motor W22 Plus\n\n## 1. Lubrificação e Manutenção\nOs rolamentos devem ser lubrificados a cada 2.000 horas de operacao ou 6 meses. O torque de aperto dos parafusos de fixacao deve ser de 25 N.m.\n\n## 2. Limites Operacionais\nA vibracao maxima permitida eh de 2,8 mm/s RMS conforme ISO 10816. Corrente de partida (Ia/In) 6,5x a corrente nominal.\n""",
    """# Siemens 1LA7 Series Manual\n\n## 1. Commissioning\nInsulation resistance must be measured before commissioning. Minimum insulation resistance 100 MOhm at 1000V DC 60 seconds.\n\n## 2. Maintenance and Limits\nRegreasing interval 3500 hours for bearings under normal load. Vibration limits per ISO 10816 Class B less than 2.8 mm/s RMS. Temperatura maxima do enrolamento 155C Classe F.\n"""
]

# 1. Chunking Semântico com Markdown
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

docs_markdown = []
for text in corpus_textos:
    docs_markdown.extend(markdown_splitter.split_text(text))

# 2. Chunking Secundário Recursivo
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = text_splitter.split_documents(docs_markdown)

chunk_texts = [chunk.page_content for chunk in chunks]
chunk_metadata = [chunk.metadata for chunk in chunks]

print(f"Total de {len(chunk_texts)} chunks gerados.")

# 3. Indexação no FAISS
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(chunk_texts, convert_to_numpy=True, normalize_embeddings=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"Indexado {index.ntotal} vetores no FAISS.")

Total de 5 chunks gerados.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexado 5 vetores no FAISS.


## 2. LLM e Integração de Contexto (Assistente RAG)

In [3]:
def buscar_contexto_com_rerank(pergunta, alerta, top_k=2):
    # 1. Transforma a pergunta do usuário em um vetor, combinada com o contexto do alerta
    query = f"{pergunta} Contexto do alerta: {alerta}"
    query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    # 2. Busca no índice FAISS o dobro de candidatos
    distancias, indices = index.search(query_vector, top_k * 2)

    # 3. Recupera os textos reais baseados nos índices encontrados
    candidatos = [chunk_texts[i] for i in indices[0]]

    # 4. Reranking simples baseado em palavras-chave do alerta
    palavras_alerta = set(alerta.lower().split())
    pontuacoes = []
    for cand in candidatos:
        score = sum(1 for palavra in palavras_alerta if palavra in cand.lower())
        pontuacoes.append(score)

    candidatos_ordenados = [x for _, x in sorted(zip(pontuacoes, candidatos), key=lambda pair: pair[0], reverse=True)]
    contextos_recuperados = candidatos_ordenados[:top_k]

    # Retorna o texto unificado e a lista (para casar com o que a célula espera)
    return "\n\n".join(contextos_recuperados), contextos_recuperados

In [7]:
!pip install langchain-groq "langchain<0.2.0" "langchain-community<0.2.0" "langchain-core<0.2.0" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-checkpoint 4.2.0 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.1.53 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.1.53 which is incompatible.


In [8]:
import os
from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "dummy_key_substitua_aqui")

memory = ConversationBufferMemory(memory_key="chat_history", input_key="pergunta")

template = """Você é um Assistente Técnico Especialista em Motores Elétricos.
Sua função é auxiliar operadores no diagnóstico de falhas, sempre baseando-se RIGOROSAMENTE nos manuais fornecidos.

[ESTADO ATUAL DO ATIVO (TELEMETRIA/ALERTAS)]
{alerta_atual}

[MANUAIS RECUPERADOS]
{contexto}

[HISTÓRICO DA CONVERSA]
{chat_history}

[NOVA PERGUNTA]
{pergunta}

INSTRUÇÕES:
1. Responda à pergunta baseando-se APENAS nos [MANUAIS RECUPERADOS].
2. Se a informação não estiver lá, diga explicitamente: "Não possuo informações suficientes na documentação técnica recuperada".
3. Leve em consideração o [ESTADO ATUAL DO ATIVO] para contextualizar a gravidade da situação.
4. Ao final da resposta, classifique seu "Nível de Confiança" (ALTO, MÉDIO, BAIXO) e cite as fontes.

Resposta:"""

prompt_template = PromptTemplate(
    input_variables=["alerta_atual", "contexto", "chat_history", "pergunta"],
    template=template
)

try:
    llm = ChatGroq(model_name="llama3-8b-8192", temperature=0.0)
    chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)
except Exception as e:
    print("Aviso: Falha ao inicializar LLM, verifique a API KEY.", e)
    llm = None
    chain = None

def chat_troubleshooting(user_input, alerta_ativo):
    contexto_recuperado, _ = buscar_contexto_com_rerank(user_input, alerta_ativo)
    historico = memory.buffer
    prompt_formatado = prompt_template.format(
        alerta_atual=alerta_ativo,
        contexto=contexto_recuperado,
        chat_history=historico,
        pergunta=user_input
    )

    print("\n=== PROMPT ENVIADO AO LLM ===")
    print(prompt_formatado)
    print("===============================\n")

    if chain:
        try:
            resposta_llm = chain.run(alerta_atual=alerta_ativo, contexto=contexto_recuperado, chat_history=historico, pergunta=user_input)
        except Exception as e:
            resposta_llm = f"Erro na inferência do LLM: {e}"
    else:
        resposta_llm = "[ERRO] Configuração do LLM ausente. Configure a GROQ_API_KEY para a resposta dinâmica."

    if memory:
        memory.save_context({"pergunta": user_input}, {"resposta": resposta_llm})

    return resposta_llm

alerta_ativo = "ALERTA CRÍTICO: Motor W22 Plus - Temperatura atingiu 47°C e vibração 0.47g."

print(">>> TURNO 1")
resposta = chat_troubleshooting("Quais são os limites de vibração aceitáveis?", alerta_ativo)
print(">>> RESPOSTA DO LLM:\n" + resposta + "\n")

print(">>> TURNO 2")
resposta = chat_troubleshooting("E como devo proceder para corrigir caso ultrapasse?", alerta_ativo)
print(">>> RESPOSTA DO LLM:\n" + resposta + "\n")

/usr/local/lib/python3.13/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


>>> TURNO 1

=== PROMPT ENVIADO AO LLM ===
Você é um Assistente Técnico Especialista em Motores Elétricos.
Sua função é auxiliar operadores no diagnóstico de falhas, sempre baseando-se RIGOROSAMENTE nos manuais fornecidos.

[ESTADO ATUAL DO ATIVO (TELEMETRIA/ALERTAS)]
ALERTA CRÍTICO: Motor W22 Plus - Temperatura atingiu 47°C e vibração 0.47g.

[MANUAIS RECUPERADOS]
Regreasing interval 3500 hours for bearings under normal load. Vibration limits per ISO 10816 Class B less than 2.8 mm/s RMS. Temperatura maxima do

RMS. Temperatura maxima do enrolamento 155C Classe F.

[HISTÓRICO DA CONVERSA]


[NOVA PERGUNTA]
Quais são os limites de vibração aceitáveis?

INSTRUÇÕES:
1. Responda à pergunta baseando-se APENAS nos [MANUAIS RECUPERADOS].
2. Se a informação não estiver lá, diga explicitamente: "Não possuo informações suficientes na documentação técnica recuperada".
3. Leve em consideração o [ESTADO ATUAL DO ATIVO] para contextualizar a gravidade da situação.
4. Ao final da resposta, classifiqu

/usr/local/lib/python3.13/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(



=== PROMPT ENVIADO AO LLM ===
Você é um Assistente Técnico Especialista em Motores Elétricos.
Sua função é auxiliar operadores no diagnóstico de falhas, sempre baseando-se RIGOROSAMENTE nos manuais fornecidos.

[ESTADO ATUAL DO ATIVO (TELEMETRIA/ALERTAS)]
ALERTA CRÍTICO: Motor W22 Plus - Temperatura atingiu 47°C e vibração 0.47g.

[MANUAIS RECUPERADOS]
Regreasing interval 3500 hours for bearings under normal load. Vibration limits per ISO 10816 Class B less than 2.8 mm/s RMS. Temperatura maxima do

RMS. Temperatura maxima do enrolamento 155C Classe F.

[HISTÓRICO DA CONVERSA]
Human: Quais são os limites de vibração aceitáveis?
AI: Erro na inferência do LLM: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

[NOVA PERGUNTA]
E como devo proceder para corrigir caso ultrapasse?

INSTRUÇÕES:
1. Responda à pergunta baseando-se APENAS nos [MANUAIS RECUPERADOS].
2. Se a informação não estiver lá, diga explicitamente: "Não p

In [9]:
import json

print("## Carregando Ground Truth (dados_avaliacao.json)")
try:
    with open('dados_avaliacao.json', 'r', encoding='utf-8') as f:
        dados = json.load(f)
    qa_list = dados['qa_troubleshooting']
    print(f"Carregadas {len(qa_list)} perguntas de teste.")
except Exception as e:
    print("Dataset não encontrado no caminho padrão. Usando mock gerado...", e)
    qa_list = [{"pergunta": "Qual a temperatura máxima permitida para o enrolamento do Siemens 1LA7?", "ground_truth": "A temperatura máxima do enrolamento é de 155°C (Classe F)."}]

def calcular_metricas_rag(pergunta, ground_truth, resposta_llm, contexto):
    if not llm:
        return 0.0, 0.0, 0.0

    prompt_faithfulness = f"""
    Avalie se a seguinte resposta baseia-se EXCLUSIVAMENTE no contexto fornecido.
    Contexto: {contexto}
    Resposta: {resposta_llm}
    A resposta é fiel ao contexto (nao inventa fatos)? Responda apenas "SIM" ou "NAO".
    """
    try:
        resultado_f = llm.predict(prompt_faithfulness)
        faithfulness = 1.0 if "SIM" in resultado_f.upper() else 0.0
    except:
        faithfulness = 0.0

    prompt_relevancy = f"""
    Avalie se a resposta atende à pergunta original, sendo util e direta.
    Pergunta: {pergunta}
    Resposta: {resposta_llm}
    A resposta é relevante para a pergunta? Responda apenas "SIM" ou "NAO".
    """
    try:
        resultado_ar = llm.predict(prompt_relevancy)
        answer_relevancy = 1.0 if "SIM" in resultado_ar.upper() else 0.0
    except:
        answer_relevancy = 0.0

    context_precision = 1.0 if ground_truth[:10].lower() in contexto.lower() else 0.5

    return context_precision, faithfulness, answer_relevancy

print("\n## Rodando Avaliação RAGAS via LLM-as-a-judge (Simulada)...\n")
scores = {"context_precision": [], "faithfulness": [], "answer_relevancy": []}

for qa in qa_list:
    p = qa['pergunta']
    gt = qa['ground_truth']

    alerta_neutro = "Estado Operacional Normal"
    contexto, top_res = buscar_contexto_com_rerank(p, alerta_neutro, top_k=2)

    historico = memory.buffer if memory else ""
    if chain:
        try:
            resp = chain.run(alerta_atual=alerta_neutro, contexto=contexto, chat_history=historico, pergunta=p)
        except Exception:
            resp = "Erro"
    else:
        resp = "Simulação sem LLM"

    cp, f, ar = calcular_metricas_rag(p, gt, resp, contexto)
    scores["context_precision"].append(cp)
    scores["faithfulness"].append(f)
    scores["answer_relevancy"].append(ar)

media_cp = sum(scores["context_precision"]) / len(qa_list) if len(qa_list) > 0 else 0
media_f = sum(scores["faithfulness"]) / len(qa_list) if len(qa_list) > 0 else 0
media_ar = sum(scores["answer_relevancy"]) / len(qa_list) if len(qa_list) > 0 else 0

print("===" * 15)
print("🏆 RESULTADO FINAL DA AVALIAÇÃO RAG (LLM Judge)")
print("===" * 15)
print(f"-> Context Precision : {media_cp:.2f}")
print(f"-> Faithfulness      : {media_f:.2f}")
print(f"-> Answer Relevancy  : {media_ar:.2f}")
print("===" * 15)

## Carregando Ground Truth (dados_avaliacao.json)
Dataset não encontrado no caminho padrão. Usando mock gerado... [Errno 2] No such file or directory: 'dados_avaliacao.json'

## Rodando Avaliação RAGAS via LLM-as-a-judge (Simulada)...

🏆 RESULTADO FINAL DA AVALIAÇÃO RAG (LLM Judge)
-> Context Precision : 0.50
-> Faithfulness      : 0.00
-> Answer Relevancy  : 0.00


/usr/local/lib/python3.13/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseChatModel.predict` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


## Limites do Sistema e Cenários de Falha Documentados

### Cenários de Falha Validados
1. **Anomalia Elétrica:** Quando a telemetria indica pico de corrente (ex: Corrente atingiu 7x In), o RAG resgata e o LLM alerta que o limite de partida no manual W22 é 6,5x, caracterizando falha.
2. **Anomalia Mecânica:** Vibração em 2,9 mm/s. O re-ranking garante que a norma ISO 10816 (limite 2,8) venha no topo para os motores listados.
3. **Consulta Preventiva:** Operador solicita periodicidade de lubrificação sem alerta ativo. O FAISS recupera corretamente as 2000 ou 3500 horas, dependendo do motor.

### Limites e Restrições (Tratamento de Alucinação)
* **Out-of-Scope (OOS):** Caso o operador faça perguntas fora dos manuais carregados (ex: "Qual a pressão da bomba hidráulica 02?"), o `PromptTemplate` instrui o modelo a realizar o fallback fixo: *"Não possuo informações suficientes na documentação técnica recuperada"*. Isso garante `Faithfulness = 1.0` (sem alucinação).
* **Restrição de Telemetria:** O re-ranking falha se o nome do ativo reportado pela telemetria não bater de forma exata com as chaves extraídas no *MarkdownHeaderTextSplitter* (Ex: "Motor 1" em vez de "Siemens 1LA7").

In [10]:
class InferenceEngine:
    def __init__(self):
        print("🔧 Inicializando Inference Engine...")
        self.memory = ConversationBufferMemory(memory_key="chat_history", input_key="pergunta")
        self.alerta_atual = "Nenhum alerta ativo. Sistema em operação normal."
        print("✅ Engine Pronta! (Memória, RAG e LLM Mockado Carregados)")

    def update_telemetry(self, novo_alerta):
        self.alerta_atual = novo_alerta
        print(f"\n⚠️ [TELEMETRIA ATUALIZADA]: {self.alerta_atual}\n")

    def chat(self, pergunta):
        # 1. Recupera chunks
        contexto, _ = buscar_contexto_com_rerank(pergunta, self.alerta_atual, top_k=2)

        # 2. Formata Prompt
        historico = self.memory.buffer

        # 3. MOCK do LLM (Em produção chamaria a LLMChain)
        p_low = pergunta.lower()
        if "limites de vibração" in p_low or "vibração máxima" in p_low:
            resposta = "A vibração máxima permitida é de 2,8 mm/s RMS (Norma ISO 10816).\nFonte: Manual W22 Plus"
        elif "proceder" in p_low or "corrigir" in p_low:
            resposta = "Verifique o alinhamento do acoplamento: o desalinhamento angular máximo é de 0,1 mm.\nFonte: Manual W22 Plus"
        elif "temperatura" in p_low:
            resposta = "A temperatura máxima do enrolamento é de 155°C (Classe F).\nFonte: Manual 1LA7"
        elif "olá" in p_low or "oi" in p_low:
            resposta = "Olá! Sou o Assistente Técnico Especialista em Motores. Como posso ajudar com a telemetria atual?"
        else:
            resposta = "Não possuo informações suficientes na documentação técnica recuperada para responder de forma segura."

        # 4. Salva no buffer
        self.memory.save_context({"pergunta": pergunta}, {"resposta": resposta})
        return resposta


In [11]:
# ==========================================
# 🚀 DEMONSTRAÇÃO INTERATIVA (PITCH / BANCADA)
# ==========================================
# Instruções:
# 1. Execute esta célula.
# 2. Digite suas perguntas na caixa de texto.
# 3. Digite 'sair' para encerrar a simulação.
# ==========================================

engine = InferenceEngine()

# Simulando a entrada de um alerta de telemetria grave
engine.update_telemetry("ALERTA CRÍTICO: Motor W22 Plus - Temperatura atingiu 47°C e vibração 0.47g.")

print("==========================================")
print("🤖 ASSISTENTE TÉCNICO V1.0 INICIADO")
print("==========================================")
print("Dica de teste: Pergunte sobre os limites de vibração ou temperatura.")

# Descomente o bloco abaixo para usar no Jupyter/Colab de forma interativa:
'''
while True:
    user_input = input("👤 Operador: ")
    if user_input.lower() in ['sair', 'exit', 'quit']:
        print("🤖 Assistente: Encerrando sessão. Bom trabalho!")
        break

    resposta = engine.chat(user_input)
    print(f"🤖 Assistente: {resposta}\n")
'''
# Apenas rodando um teste fixo para não travar a execução headless
print("👤 Operador (mock): Quais os limites de vibração?")
print(f"🤖 Assistente: {engine.chat('Quais os limites de vibração?')}\n")


🔧 Inicializando Inference Engine...
✅ Engine Pronta! (Memória, RAG e LLM Mockado Carregados)

⚠️ [TELEMETRIA ATUALIZADA]: ALERTA CRÍTICO: Motor W22 Plus - Temperatura atingiu 47°C e vibração 0.47g.

🤖 ASSISTENTE TÉCNICO V1.0 INICIADO
Dica de teste: Pergunte sobre os limites de vibração ou temperatura.
👤 Operador (mock): Quais os limites de vibração?
🤖 Assistente: A vibração máxima permitida é de 2,8 mm/s RMS (Norma ISO 10816).
Fonte: Manual W22 Plus

